In [ ]:
import os
import zipfile
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import cv2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


# Path to the uploaded dataset zip file
zip_path = '/content/dataset-resized.zip'

# Temporary extraction path
extraction_path = 'path_to_extraction_folder'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

# Define the path to the dataset folder
dataset_folder = os.path.join(extraction_path, "dataset-resized")

# List of categories (folder names)
categories = [dir_name for dir_name in os.listdir(dataset_folder) if not dir_name.startswith('.')]

# Initialize lists to store image data and labels
data = []
labels = []

# Define desired size of images (example: 128x128 pixels)
image_size = (128, 128)

# Loop through each category
for category in categories:
    category_folder = os.path.join(dataset_folder, category)
    print(f"Processing category: {category}")

    # Loop through each image file in the category folder
    for filename in tqdm(os.listdir(category_folder), desc=f"Loading {category}"):
        image_path = os.path.join(category_folder, filename)

        # Open the image using PIL and resize it
        with Image.open(image_path) as img:
            img_resized = img.resize(image_size)

        # Convert the image to a numpy array and normalize it
        image_array = np.array(img_resized) / 255.0

        # Append the processed image data and label to the lists
        data.append(image_array)
        labels.append(category)

# Convert lists to numpy arrays for machine learning processing
data = np.array(data)
labels = np.array(labels)

# Convert labels to numerical values using LabelEncoder
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, encoded_labels, test_size=0.2, random_state=42)

# You can now proceed with training your machine learning model
print("Data and labels prepared. Ready for model training.")


In [ ]:
import os
import numpy as np
import pandas as pd
from skimage.io import imread
from skimage.transform import resize
from tqdm import tqdm

# Corrected path to the dataset folder after extraction
base_path = '/content/path_to_extraction_folder/dataset-resized'  # Adjusted path

# Categories of the images
Categories = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Initialize lists to store flattened image data and corresponding labels
flat_data_arr = []
target_arr = []

# Loop through each category
for i in Categories:
    print(f'Loading... category : {i}')
    path = os.path.join(base_path, i)

    # Ensure the path exists before processing
    if os.path.exists(path):
        # Loop through each image file in the category folder
        for img in tqdm(os.listdir(path), desc=f'Loading {i}'):
            img_path = os.path.join(path, img)

            # Read the image, resize it to 150x150 pixels, and normalize the pixel values
            img_array = imread(img_path)
            img_resized = resize(img_array, (150, 150, 3))

            # Flatten the resized image and append it to the list, along with the label
            flat_data_arr.append(img_resized.flatten())
            target_arr.append(Categories.index(i))

        print(f'Loaded category: {i} successfully')
    else:
        print(f"Path does not exist: {path}")

# Convert lists to numpy arrays for further processing
flat_data = np.array(flat_data_arr)
target = np.array(target_arr)

# Create a DataFrame with the flattened image data
df = pd.DataFrame(flat_data)

# Add the target labels to the DataFrame
df['Target'] = target
df


In [ ]:
import cv2
from sklearn.metrics import confusion_matrix
# Function to extract color histograms from images
def extract_color_histogram(image_path, bins=32):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hist = [cv2.calcHist([image], [i], None, [bins], [0, 256]) for i in range(3)]
    hist = np.concatenate([cv2.normalize(h, h).flatten() for h in hist])
    return hist
# Function to process images and extract features
def process_images(base_path):
    features = []
    labels = []
    categories = os.listdir(base_path)
    for category in categories:
        if category == ".DS_Store":
            continue
        category_path = os.path.join(base_path, category)
        images = [os.path.join(category_path, img) for img in os.listdir(category_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for image_path in images:
            hist_features = extract_color_histogram(image_path)
            features.append(hist_features)
            labels.append(category)
    return np.array(features), np.array(labels)
# Extract features and labels from the dataset
features, labels = process_images(base_path)
# Convert labels to numerical values
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, encoded_labels, test_size=0.2, random_state=42)
# Initialize the Logistic Regression classifier
log_reg_clf = LogisticRegression(max_iter=1000, random_state=42)
log_reg_clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = log_reg_clf.predict(X_test)
# Calculate accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
print("Accuracy:", accuracy)
print("Classification Report:\n", report)
# Display the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
from sklearn.decomposition import PCA
# Apply PCA to reduce the feature dimensionality
pca = PCA(n_components=0.95)  # Adjust this to the desired variance ratio
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
# Initialize the Logistic Regression classifier
log_reg_clf = LogisticRegression(max_iter=1000, random_state=42)
log_reg_clf.fit(X_train_pca, y_train)
# Make predictions on the test set with PCA applied
y_pred_pca = log_reg_clf.predict(X_test_pca)
# Calculate accuracy and classification report
accuracy_pca = accuracy_score(y_test, y_pred_pca)
report_pca = classification_report(y_test, y_pred_pca, target_names=label_encoder.classes_)
print("PCA Accuracy:", accuracy_pca)
print("PCA Classification Report:\n", report_pca)
# Display the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))



In [ ]:
from sklearn.metrics import confusion_matrix

# Calculate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Calculate sensitivity and specificity for each class
sensitivity_per_class = []
specificity_per_class = []
for i in range(conf_matrix.shape[0]):
    true_positives = conf_matrix[i, i]
    false_negatives = np.sum(conf_matrix[i, :]) - true_positives
    false_positives = np.sum(conf_matrix[:, i]) - true_positives
    true_negatives = np.sum(conf_matrix) - true_positives - false_negatives - false_positives

    sensitivity = true_positives / (true_positives + false_negatives)
    specificity = true_negatives / (true_negatives + false_positives)

    sensitivity_per_class.append(sensitivity)
    specificity_per_class.append(specificity)

# Calculate average sensitivity and specificity
average_sensitivity = np.mean(sensitivity_per_class)
average_specificity = np.mean(specificity_per_class)

print("Average Sensitivity:", average_sensitivity)
print("Average Specificity:", average_specificity)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
# Function to extract both RGB and HSV color histograms from images
def extract_features(image_path, bins=32):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # Convert to HSV
    # Calculate the histogram for RGB
    rgb_hist = np.concatenate([cv2.calcHist([image], [i], None, [bins], [0, 256]).flatten() for i in range(3)])
    # Calculate the histogram for HSV
    hsv_hist = np.concatenate([cv2.calcHist([hsv_image], [i], None, [bins], [0, 256]).flatten() for i in range(3)])
    # Concatenate RGB and HSV histograms into a single feature vector
    feature_vector = np.concatenate((rgb_hist, hsv_hist))
    return feature_vector
# Function to process images and extract features
def process_images(base_path, bins=32):
    features = []
    labels = []
    categories = os.listdir(base_path)
    for category in categories:
        if category == ".DS_Store" or category == '__MACOSX':
            continue
        category_path = os.path.join(base_path, category)
        images = [os.path.join(category_path, img) for img in os.listdir(category_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for image_path in images:
            feature_vector = extract_features(image_path, bins)
            features.append(feature_vector)
            labels.append(category)
    return np.array(features), np.array(labels)
# Extract features and labels from the dataset
features, labels = process_images(base_path)
# Convert labels to numerical values
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, encoded_labels, test_size=0.2, random_state=42)
# Create a pipeline that includes scaling and logistic regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic_regression', LogisticRegression(max_iter=1000))
])
# Define a range of hyperparameters for tuning
parameters = {
    'logistic_regression__C': np.logspace(-4, 4, 20),
    'logistic_regression__class_weight': [None, 'balanced'],
    'logistic_regression__solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
}
# Initialize GridSearchCV with the pipeline and hyperparameter space
grid_search = GridSearchCV(pipeline, parameters, cv=5, verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)
# Retrieve the best hyperparameters and the corresponding best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
# Make predictions using the best model
y_pred = best_model.predict(X_test)
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
# Print the results
print(f"Best Parameters: {best_params}")
print("Accuracy:", accuracy)
print("Classification Report:\n", report)
# Display the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import os
import pickle

# Assuming 'pipeline' is your trained model and 'metrics_results' is your computed metrics.

# Define the path for the combined pickle file
combined_file_path = '/content/combined_model_and_metrics.pkl'

# Create the directory if it does not exist
os.makedirs(os.path.dirname(combined_file_path), exist_ok=True)

# Combine the model and metrics into a dictionary
combined_data = {
    'model': pipeline,
    'metrics': classification_report
}

# Save the combined data to a single pickle file
with open(combined_file_path, 'wb') as combined_file:
    pickle.dump(combined_data, combined_file)

print(f'Combined model and metrics are saved at {combined_file_path}.')

